In [ ]:
# """Boundary information for Jefferson County, KY, USA"""

# LONGITUDES = LONG_MIN, LONG_MAX = -85.94712712079293, -85.40492183942492
# LATITUDES = LAT_MIN, LAT_MAX = 37.99712528351634, 38.38023822809115

# DELTA_LONG = LONG_MAX - LONG_MIN  # == 0.5422052813680125
# DELTA_LAT = LAT_MAX - LAT_MIN  # == 0.3831129445748118

In [19]:
import os

import json
import pandas as pd

SOURCE = "/Users/bencampbell/code/county_coverage/data/raw/Louisville_Metro_KY_County_Boundaries.geojson"
assert os.path.exists(SOURCE)

In [ ]:
approx_latitude_span = 26 #mi
approx_longitude_span = 30 #mi
feet_in_a_mile = 5280 # I can never remember this

Source data for county boundary is a GeoJSON file that looks like this:

```json
{
"type": "FeatureCollection",
"name": "Louisville_Metro_KY_County_Boundaries",
"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
"features": [...]}
```

Where each item in `"features"` represent a county in the Louisville, KY metro area. This includes Jefferson County, where Louisville is, and several surrounding counties. Each `feature` or county object looks like this:

```json
{ "type": "Feature", 
  "properties": { "OBJECTID": 7, 
                  "CNTY_NAME": "JEFFERSON", 
                  "FIPS": "21111", 
                  "STATE_FIPS": "21", 
                  "CNTY_FIPS": "111", 
                  "SHAPEAREA": 11083783720.6446, 
                  "SHAPELEN": 513054.30366378697 }, 
  "geometry": { "type": "Polygon", 
                "coordinates": [ [ [ -85.575811868593064, 38.334545868470848 ], 
                                   [ -85.578070603648868, 38.335714204871564 ], 
                                   [ -85.579003771392806, 38.336193264774884 ],
                                    ... ] ] } }
```

Jefferson County is the one we are interested in here. The value for `"coordinates"` is a list of lists. Each interior list contains points defining a polygon that represents the boundary of the county. The points are listed as `[longitude, latitude]` pairs. The `coordinates` are really all we need, once we find the correct county object in the list of `features`.

In [26]:
with open(SOURCE, 'r') as file:
    data = json.load(file)

counties = data['features']
# Each feature represents a county boundary. Need to find the right one.

#len(data['features']) # == 12

# The county I am interested in is called Jefferson.

for county in counties:
    if county['properties']['CNTY_NAME'] == 'JEFFERSON':
        JEFFCO = county
        break

geo = JEFFCO['geometry']
# Geo has two properties: 'type' and 'coordinates'. We only need the latter.
# 'type' is Polygon
coordinates = geo['coordinates']

len(coordinates) # == 1
# Sometimes Polygon geometry can consist of more than one shape.
# In this case, the county boundary is just one shape.

coordinates = coordinates[0]
len(coordinates) # == 2831 points



def sort_long_lat(points:list, *, name:str) -> pd.Series:
    """Create bounding box from a list of (longitude, latitude) points."""
    longitudes = set()
    latitudes = set()
    for long, lat in points:
        longitudes.add(long)
        latitudes.add(lat)

    return pd.Series(data={"west_longitude": min(longitudes), "east_longitide": max(longitudes),
                           "south_latitude": min(latitudes), "north_latitude": max(latitudes)},
                     name=name)

CO_BOUNDARY = sort_long_lat(coordinates, name="CO_BOUNDARY")
CO_BOUNDARY

west_longitude   -85.947127
east_longitide   -85.404922
south_latitude    37.997125
north_latitude    38.380238
Name: CO_BOUNDARY, dtype: float64

The official bounding box for Jefferson County, KY:

| boundary | value | 
|----------|-------|
| West longitude | -85.94712712079293 |
| East longitide | -85.40492183942492 |
| South_latitude | 37.99712528351634 |
| North_latitude | 38.38023822809115 |

In [ ]:
# TODO Get similar data from centerlines to see if any of the roads exit county boundaries. 

CENTERLINES = "../../data/cleaner/centerlines_raw.csv"
assert os.path.exists(CENTERLINES)

import pandas as pd
cl = pd.read_csv(CENTERLINES, low_memory=False)
coords = cl['COORDINATES'].to_list()


AssertionError: 

In [ ]:

#coords[0]
# Each cooordinate is a string.
# Like this: '-85.6809503218278,38.1588670887491 -85.6810007757001,38.158161039538 -85.681234634906,38.1581804844049'

list(map(float, coords[4].split()[0].split(','))) # Works

def get_points():
    for coordinate_string in coords:
        for point in coordinate_string.split():
            yield map(float, point.split(','))

longitudes, latitudes = sort_long_lat(get_points())


clong_max = max(longitudes)
clong_min = min(longitudes)
clat_max = max(latitudes)
clat_min = min(latitudes)

In [ ]:
# TODO get boundaries for intersections and compare